# 05.6 The `None` Type

`None` represents the **absence of a value**. It is Python's answer to `null`,
`nil` and `undefined` — but with one important difference: there is only ever
**one** `None` object in the entire program.

## Theory

### A singleton

`None` is the sole instance of `NoneType`. Every reference to `None` anywhere in
your program points at the same object:

```python
a = None
b = None
a is b        # True - always
```

That is **why** the idiom is `x is None` rather than `x == None`. Identity is
guaranteed; equality could be overridden by a class.

You cannot create another one — `NoneType()` returns the existing instance.

### Where `None` appears

1. **The default return value.** A function with no `return` returns `None`.
2. **Sentinel defaults.** `def f(x=None)` — the fix for the mutable default trap.
3. **Missing data.** `dict.get()` returns `None` when a key is absent.
4. **Uninitialised state.** A variable that will be set later.
5. **Signalling failure** — though raising an exception is usually better.

### `None` versus `0` versus `""` versus `False`

These mean genuinely different things, and conflating them causes real bugs:

| Value | Means |
|---|---|
| `None` | No value at all; unknown |
| `0` | A quantity, which happens to be zero |
| `""` | A string, which happens to be empty |
| `False` | A boolean answer, which is negative |

All four are falsy, which is exactly why `if not x:` cannot distinguish them.

### Mutating methods return `None`

Python follows a convention: a method that **modifies in place** returns `None`,
so you cannot accidentally chain it and lose your data.

```python
items = [3, 1, 2]
items = items.sort()     # items is now None - the classic bug
```

In [ ]:
# None is a singleton - there is only ever one.
first = None
second = None

print("first is second:", first is second)
print("id(first):      ", id(first))
print("id(second):     ", id(second))
print("type(None):     ", type(None).__name__)

# NoneType() returns the same object rather than creating a new one.
none_type = type(None)
print("")
print("NoneType() is None:", none_type() is None)

# You cannot subclass it.
try:
    class MyNone(none_type):
        pass
except TypeError as error:
    print("Subclassing NoneType:", error)

print("")
print("This guarantee is why `x is None` is the correct idiom.")

In [ ]:
# Why `is` rather than `==`.
value = None

print("Both work for plain None:")
print("   value is None:", value is None)
print("   value == None:", value == None)


class Deceptive:
    """A class claiming equality with anything, including None."""

    def __eq__(self, other):
        return True


tricky = Deceptive()

print("")
print("But a class can lie about equality:")
print("   tricky == None:", tricky == None, "<- wrong answer")
print("   tricky is None:", tricky is None, "<- identity cannot be faked")

print("")
print("Linters flag `== None` as E711 for exactly this reason.")

## Where `None` turns up

In [ ]:
# 1. The implicit return value.
def no_return_statement():
    """Do something without returning anything."""
    result = 1 + 1


def bare_return():
    """Return early with no value."""
    return


print("1. Implicit returns:")
print("   function with no return:", no_return_statement())
print("   function with bare return:", bare_return())

# print() itself returns None.
print("   print() returns:", repr(print()))

# 2. dict.get on a missing key.
settings = {"host": "localhost"}
print("")
print("2. Missing dictionary keys:")
print("   settings.get('host'):", settings.get("host"))
print("   settings.get('port'):", settings.get("port"))
print("   settings.get('port', 8080):", settings.get("port", 8080))

# 3. Sentinel defaults - the fix from Chapter 04.
def add_item(item, target=None):
    """Create a fresh list when none is supplied."""
    if target is None:
        target = []
    target.append(item)
    return target


print("")
print("3. Sentinel default:")
print("   call 1:", add_item("a"))
print("   call 2:", add_item("b"), "<- independent")

# 4. re.match returns None when there is no match.
import re
print("")
print("4. Search functions:")
print("   re.match('a', 'abc'):", re.match("a", "abc"))
print("   re.match('z', 'abc'):", re.match("z", "abc"))

## The mutating-method trap

Methods that change an object in place return `None`. Assigning their result
destroys your data.

In [ ]:
# THE BUG.
numbers = [3, 1, 2]
numbers = numbers.sort()

print("THE BUG:")
print("   numbers = numbers.sort() ->", numbers, "<- the list is gone")

# THE FIX - two options.
numbers = [3, 1, 2]
numbers.sort()
print("")
print("FIX 1 - sort in place, do not assign:")
print("   numbers.sort(); numbers ->", numbers)

numbers = [3, 1, 2]
sorted_numbers = sorted(numbers)
print("")
print("FIX 2 - use sorted(), which returns a new list:")
print("   sorted(numbers) ->", sorted_numbers)
print("   original:", numbers, "<- unchanged")

# Every in-place method behaves this way.
print("")
print("Methods returning None because they mutate:")

sample = [1, 2, 3]
operations = [
    ("list.append", sample.append(4)),
    ("list.extend", sample.extend([5])),
    ("list.reverse", sample.reverse()),
    ("list.sort", sample.sort()),
]

for name, result in operations:
    print(f"   {name:<16} returns {result!r}")

sample_dict = {"a": 1}
print(f"   {'dict.update':<16} returns {sample_dict.update({'b': 2})!r}")

sample_set = {1}
print(f"   {'set.add':<16} returns {sample_set.add(2)!r}")

print("")
print("The convention is deliberate: it prevents you from chaining a")
print("mutation and silently losing the object.")

## Distinguishing `None` from other falsy values

In [ ]:
def classify(value):
    """Report which falsy category a value belongs to."""
    if value is None:
        return "None - no value at all"
    if value is False:
        return "False - a negative boolean"
    if value == 0 and isinstance(value, (int, float)):
        return "zero - a real quantity"
    if value == "":
        return "empty string - a real string"
    if not value:
        return "empty collection"
    return "truthy"


test_values = [None, False, 0, 0.0, "", [], {}, "text", 5]

print("Value        Classification")
print("-" * 48)
for value in test_values:
    print(f"{repr(value):<12} {classify(value)}")

print("")
print("`if not x:` would treat every one of the first seven identically.")
print("Only explicit checks can tell them apart.")

In [ ]:
# A sentinel for when None itself is a valid value.
# This is the standard technique when you must distinguish
# "not supplied" from "supplied as None".

MISSING = object()


def get_setting(config, key, default=MISSING):
    """Look up a key, distinguishing 'no default' from 'default is None'."""
    if key in config:
        return config[key]

    if default is MISSING:
        raise KeyError(f"{key!r} not found and no default given")

    return default


config = {"timeout": None, "retries": 3}

print("Config:", config)
print("")
print("   get_setting(config, 'retries')      ->", get_setting(config, "retries"))
print("   get_setting(config, 'timeout')      ->", get_setting(config, "timeout"))
print("   get_setting(config, 'missing', None)->", get_setting(config, "missing", None))

try:
    get_setting(config, "missing")
except KeyError as error:
    print("   get_setting(config, 'missing')      ->", error)

print("")
print("A unique object() is the standard sentinel. It cannot collide")
print("with any real value, because nothing else has its identity.")

## `None` and type hints

`Optional[X]` means "X or None". Modern syntax writes it `X | None`.

In [ ]:
from typing import Optional

# Both of these mean the same thing.
def find_user_old(user_id: int) -> Optional[str]:
    """Return a name, or None when not found."""
    users = {1: "Asha", 2: "Ben"}
    return users.get(user_id)


def find_user_new(user_id: int) -> str | None:
    """Modern syntax for the same signature."""
    users = {1: "Asha", 2: "Ben"}
    return users.get(user_id)


print("Both signatures mean 'str or None':")
print("   Optional[str] ->", find_user_old.__annotations__["return"])
print("   str | None    ->", find_user_new.__annotations__["return"])

print("")
print("   find_user_new(1):", find_user_new(1))
print("   find_user_new(9):", find_user_new(9))

# A default of None almost always implies an Optional parameter.
def connect(host: str, port: int | None = None) -> str:
    """Connect, defaulting the port when not supplied."""
    if port is None:
        port = 443
    return f"connecting to {host}:{port}"


print("")
print("   connect('example.com'):      ", connect("example.com"))
print("   connect('example.com', 8080):", connect("example.com", 8080))

print("")
print("Type hints are covered fully in Chapter 32.")

## Takeaways

1. `None` is a **singleton** — one object for the whole program, so `is None` is
   always reliable.
2. Use **`x is None`**, never `x == None`. A class can override `__eq__` but not
   identity. Linters flag this as `E711`.
3. A function with no `return` returns `None`. So does a bare `return`.
4. `None` is the standard **sentinel default**, and the fix for the mutable
   default trap.
5. Methods that **mutate in place return `None`** — `items = items.sort()`
   destroys your list.
6. `None`, `0`, `""` and `False` all mean different things but are all falsy —
   only explicit checks distinguish them.
7. When `None` is itself a valid value, use a unique `object()` as the sentinel.
8. `Optional[X]` and `X | None` both mean "X or None".

## Try it yourself

1. Confirm `None is None` and that `id(None)` never changes.
2. Write a function with no `return` and print its result.
3. Run `numbers = [3,1,2].sort()` then print `numbers`. Explain.
4. Write `get(config, key, default)` where `None` is a valid default.
5. List five standard library functions returning `None` to signal "not found".